In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
from tqdm.notebook import tqdm

In [ ]:
# Directories
CODE_DIR = Path(r"D:\StockTwits\Code")
DATA_DIR = Path(r"D:\StockTwits\Data\v1\data\csv")
FIGURES_DIR = Path(r"D:\StockTwits\Figures")
MODEL_DATA_DIR = Path(r"D:\StockTwits\Data")

# Input files
INPUT_DATA = MODEL_DATA_DIR / "merged_master.pkl"

# Text-embedding variant to evaluate (see 02 - prepare training dataset/add_text_features.ipynb):
#   None -> the plain all-features predictions; "raw" | "pca" | "supervised" -> the matching
#   predictions_*_input=N_text=<variant>.pkl files written by a model notebook run with that TEXT_VARIANT.
TEXT_VARIANT = None

def find_all_features_file(model_type, text_variant=None):
    """Resolve the 'all features' prediction filename for a model type: the largest
    input-count file on disk (excluding the 2-feature baseline) whose _text=<variant> tag
    matches `text_variant` (files without a tag when text_variant is None), so this doesn't
    need updating whenever the feature set changes."""
    import re
    pattern = re.compile(r"input=(\d+)(?:_text=([A-Za-z]+))?")
    candidates = []
    for p in MODEL_DATA_DIR.glob(f"predictions_{model_type}_input=*.pkl"):
        m = pattern.search(p.stem)
        if m is None or int(m.group(1)) == 2 or m.group(2) != text_variant:
            continue
        candidates.append((int(m.group(1)), p.name))
    if not candidates:
        tag = "" if text_variant is None else f"_text={text_variant}"
        return f"predictions_{model_type}_input=NOT_FOUND{tag}.pkl"
    return max(candidates, key=lambda t: t[0])[1]

# Model prediction files
# Key = column name in the dataframe, Value = prediction filename
MODELS = {
    'lr_2': 'predictions_linear_regression_input=2.pkl',
    'lr_all': find_all_features_file('linear_regression', TEXT_VARIANT),
}

# Time range to run regressions
START_DATE = '2012-01-01'
END_DATE = '2022-12-31'

# Prediction target
TARGET = 'f_cumret1'

# Put together all predictions

In [ ]:
# Load the original aggregated tweets data
df = pd.read_pickle(INPUT_DATA)[['date', 'permno', 'ticker', TARGET, 'log_volume']].copy()
df = df[(df['date'] >= START_DATE) & (df['date'] <= END_DATE)]

for model_col, filename in MODELS.items():
    # Load model predictions
    model_predictions = pd.read_pickle(MODEL_DATA_DIR / filename).drop(columns=['index', 'ticker'])
    model_predictions.columns = ['date', 'permno', model_col]

    # Merge with master data
    # LEFT merge: a model with a coverage gap contributes NaN for those stock-days instead of
    # (as a chained inner merge would) deleting them from every other model's sample too.
    df = pd.merge(df, model_predictions, on=['date', 'permno'], how='left')

In [ ]:
# Cross-sectionally de-mean the target and predictions each day
daily_means = df.groupby('date')[[TARGET] + list(MODELS.keys())].transform('mean')
df[TARGET] = df[TARGET] - daily_means[TARGET]
for col in list(MODELS.keys()):
    df[col] = df[col] - daily_means[col]

print("De-meaned target and predictions by date")
print(f"  Mean of {TARGET} after de-meaning: {df[TARGET].mean():.2e}")
for col in list(MODELS.keys()):
    print(f"  Mean of {col} after de-meaning: {df[col].mean():.2e}")

# Run sentiment regressions

In [ ]:
reg_results = []
for pred_col in tqdm(MODELS.keys(), desc="Running regressions"):
    reg_data = df[['permno','date', TARGET, pred_col]].dropna().copy()
    reg_data['date'] = pd.to_datetime(reg_data['date'])
    reg_data['date'] = reg_data['date'].dt.year * 10000 + reg_data['date'].dt.month*100 + reg_data['date'].dt.day 
    reg_data = reg_data.rename(columns={pred_col: 'pred'})
    y = reg_data[TARGET]
    # X = sm.add_constant(reg_data['pred'])
    X = reg_data['pred']
    model = sm.OLS(y, X, missing='drop').fit(cov_type='cluster',cov_kwds={'groups':np.array(reg_data[['permno','date']])})
    reg_results.append(model)

# Render the results in a table

In [ ]:
from latex_table import linear_regression

# Format and save the table
vars_to_include = ['pred', 'const']
var_names = ["Prediction", "Const."]
rename_dict = dict(zip(vars_to_include, var_names))

tbl = linear_regression(reg_results)
tbl.rename_variables(rename_dict)
tbl.columns = MODELS.keys()
tbl.obs = True
tbl.R2 = True
tbl.float_format = ".6f"
tbl.render(midrule=True)
tbl.tbl